# Chapter 14 Companion Notebook: Foundations of Deep Learning in Business Analytics

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch14_Foundations_of_Deep_Learning.ipynb)

This notebook accompanies Chapter 14 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




Classroom note: this notebook uses synthetic customer data and compact PyTorch models. It requires no paid API, external dataset, or downloaded model weights.

Copyright 2026 to present.

## How to use this notebook

Run the cells from top to bottom. Keep `FAST_MODE = True` during a class session. The notebook begins with tensors, activations, forward computation, and automatic differentiation, then builds an end-to-end churn-risk workflow with classical baselines, multilayer perceptrons, categorical embeddings, early stopping, regularization, calibration checks, business thresholds, and leakage audits.

## Why this matters (business framing)

Deep learning is not automatically the best choice for every prediction problem. Its value appears when learned representations, nonlinear interactions, high-dimensional inputs, or multimodal data create a meaningful advantage over simpler methods. That advantage must still justify higher experimentation cost, weaker transparency, and greater maintenance effort.

This notebook treats a neural network as a business analytics system rather than as an isolated architecture. We define the prediction moment, construct only information available at that moment, establish transparent baselines, train with a validation set, preserve the best checkpoint, examine overfitting, and convert probabilities into a retention action using explicit costs and benefits. The same discipline carries into the vision, sequence, representation-learning, and generative-model chapters that follow.

## Agenda

1. Setup and reproducibility
2. Business decision contract and synthetic customer data
3. Tensors, parameter shapes, and activation functions
4. Forward passes and function composition
5. Output layers and loss functions
6. Backpropagation through automatic differentiation
7. Leakage-safe preprocessing and classical baselines
8. Multilayer perceptron training with mini-batches and early stopping
9. Categorical embeddings for high-cardinality business identifiers
10. Capacity, regularization, learning rate, and overfitting
11. Data leakage laboratory
12. Calibration, business thresholds, and slice evaluation
13. Governance, experiment records, and saved artifacts
14. Exercises

## Learning objectives (measurable)

By the end of this notebook, you should be able to describe tensor shapes through an MLP, match an output layer and loss to a regression or classification task, inspect gradients produced by automatic differentiation, train a neural network with mini-batches and early stopping, compare a neural network with classical baselines, encode a categorical identifier with an embedding, diagnose overfitting from training and validation curves, identify several leakage pathways, and select a probability threshold using business value rather than accuracy alone.

## Connection map

Chapter 4 introduced gradient descent and Chapter 5 introduced honest model evaluation. Chapter 14 combines those ideas inside neural networks. Chapter 15 applies the workflow to images. Chapter 16 applies it to sequences and attention. Chapter 17 focuses on learned representations, and Chapters 18 and 19 move from prediction to generation. Across these chapters, the stable workflow is to define the decision, protect the evaluation boundary, establish a baseline, train a representation, measure incremental value, and monitor the deployed system.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# ============================================================
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
import json
import copy
import math
import time
import random
import warnings
import importlib
import subprocess
from pathlib import Path


def ensure(pkg_import_name, pip_name=None):
    """Install a package only if it is missing."""
    try:
        importlib.import_module(pkg_import_name)
    except Exception:
        pip_target = pip_name or pkg_import_name
        print(f"Installing {pip_target} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_target])


for import_name, pip_name in [
    ("numpy", None),
    ("pandas", None),
    ("sklearn", "scikit-learn"),
    ("matplotlib", None),
    ("joblib", None),
    ("torch", None),
]:
    ensure(import_name, pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from IPython.display import display

from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 160)

SEED = 685
FAST_MODE = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def reset_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


reset_seeds()
if DEVICE == "cpu":
    torch.set_num_threads(1)

BASE_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
OUT_DIR = BASE_DIR / "baai_ch14_deep_learning_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

N_CUSTOMERS = 900 if FAST_MODE else 1600
N_MONTHS = 12
TRAIN_EPOCHS = 30 if FAST_MODE else 50
BATCH_SIZE = 256

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print({
    "FAST_MODE": FAST_MODE,
    "N_CUSTOMERS": N_CUSTOMERS,
    "N_MONTHS": N_MONTHS,
    "TRAIN_EPOCHS": TRAIN_EPOCHS,
    "BATCH_SIZE": BATCH_SIZE,
    "OUT_DIR": str(OUT_DIR),
})

## Utility functions

The helpers below keep the main sections focused on modeling and business decisions. Metrics are calculated from probabilities because ranking, calibration, and threshold selection all matter in operational classification.

In [ ]:
# ============================================================
# Utility functions
# ============================================================
def sigmoid_np(x):
    x = np.clip(np.asarray(x, dtype=float), -40, 40)
    return 1.0 / (1.0 + np.exp(-x))


def count_trainable_parameters(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def safe_auc(y_true, prob):
    try:
        return float(roc_auc_score(y_true, prob))
    except Exception:
        return np.nan


def safe_ap(y_true, prob):
    try:
        return float(average_precision_score(y_true, prob))
    except Exception:
        return np.nan


def probability_metrics(y_true, prob, threshold=0.50):
    y_true = np.asarray(y_true).astype(int)
    prob = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    pred = (prob >= threshold).astype(int)
    return {
        "roc_auc": safe_auc(y_true, prob),
        "average_precision": safe_ap(y_true, prob),
        "log_loss": float(log_loss(y_true, prob)),
        "brier_score": float(brier_score_loss(y_true, prob)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "predicted_positive_rate": float(pred.mean()),
        "threshold": float(threshold),
    }


def expected_calibration_error(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for left, right in zip(edges[:-1], edges[1:]):
        mask = (prob >= left) & (prob < right if right < 1 else prob <= right)
        if mask.any():
            ece += mask.mean() * abs(y_true[mask].mean() - prob[mask].mean())
    return float(ece)


def to_result_row(model_name, split_name, y_true, prob, threshold=0.50, fit_seconds=np.nan):
    row = {
        "model": model_name,
        "split": split_name,
        "fit_seconds": float(fit_seconds) if np.isfinite(fit_seconds) else np.nan,
    }
    row.update(probability_metrics(y_true, prob, threshold))
    return row


def make_dense_ohe():
    """Support both current and older scikit-learn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def predict_torch_logits(model, X, batch_size=1024):
    model.eval()
    outputs = []
    tensor = torch.as_tensor(X, dtype=torch.float32)
    loader = DataLoader(TensorDataset(tensor), batch_size=batch_size, shuffle=False)
    with torch.no_grad():
        for (xb,) in loader:
            outputs.append(model(xb.to(DEVICE)).detach().cpu().numpy())
    return np.concatenate(outputs)


def utility_curve(y_true, prob, tp_value=45.0, fp_cost=8.0, fn_cost=55.0):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    rows = []
    for threshold in np.linspace(0.02, 0.80, 80):
        pred = (prob >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        total_value = tp * tp_value - fp * fp_cost - fn * fn_cost
        rows.append({
            "threshold": float(threshold),
            "tp": int(tp),
            "fp": int(fp),
            "fn": int(fn),
            "tn": int(tn),
            "net_value": float(total_value),
            "net_value_per_100": float(100 * total_value / len(y_true)),
            "action_rate": float(pred.mean()),
        })
    return pd.DataFrame(rows)

## 2. Business decision contract and synthetic customer data

The unit of analysis is one customer at one monthly prediction time. The target is whether the customer will churn during the next 30 days. The intended action is a retention intervention, so false positives consume offer budget and false negatives lose customers who might have been saved.

The synthetic data include numeric behavior, categorical context, repeated customers, seasonal change, missing values, and nonlinear interactions. Two columns are intentionally created after the outcome. They are retained only for the leakage laboratory and must not enter the legitimate model.

In [ ]:
# ============================================================
# 2.1 A practical decision framework
# ============================================================
decision_framework_df = pd.DataFrame([
    {"consideration": "data type", "favor classical methods": "structured tabular data", "consider deep learning": "images, text, audio, sequences, or multimodal inputs"},
    {"consideration": "dataset size", "favor classical methods": "small or moderate sample", "consider deep learning": "large sample or high-dimensional input"},
    {"consideration": "feature engineering", "favor classical methods": "domain relationships can be encoded", "consider deep learning": "useful features are difficult to specify manually"},
    {"consideration": "interpretability", "favor classical methods": "high justification requirement", "consider deep learning": "predictive gain can justify lower transparency"},
    {"consideration": "compute and time", "favor classical methods": "limited budget and rapid deployment", "consider deep learning": "GPU access and time for experimentation"},
    {"consideration": "maintenance", "favor classical methods": "stable data and infrequent retraining", "consider deep learning": "evolving data and maintained training pipeline"},
    {"consideration": "incremental value", "favor classical methods": "baseline already meets the decision need", "consider deep learning": "small gains create material business value"},
])

task_contract_df = pd.DataFrame([
    {"component": "unit of analysis", "definition": "one customer at one monthly prediction time"},
    {"component": "target", "definition": "churn during the next 30 days"},
    {"component": "prediction moment", "definition": "start of the month, before future service and retention outcomes"},
    {"component": "action", "definition": "offer retention support to customers above a chosen risk threshold"},
    {"component": "primary model metrics", "definition": "ROC AUC, average precision, log loss, and Brier score"},
    {"component": "decision metric", "definition": "net value after intervention benefits, offer cost, and missed-churn cost"},
])

display(decision_framework_df)
display(task_contract_df)

In [ ]:
# ============================================================
# 2.2 Generate a synthetic monthly customer panel
# ============================================================
def generate_customer_panel(n_customers=N_CUSTOMERS, n_months=N_MONTHS, seed=SEED):
    rng = np.random.default_rng(seed)
    n_stores = 60
    n_products = 14

    customer_id = np.arange(n_customers)
    latent_risk = rng.normal(0, 1, n_customers)
    base_tenure = rng.integers(1, 72, n_customers)
    segment = rng.choice(
        ["value", "mainstream", "premium", "business"],
        n_customers,
        p=[0.28, 0.42, 0.20, 0.10],
    )
    home_store = rng.integers(0, n_stores, n_customers)
    base_spend = rng.lognormal(np.log(70), 0.35, n_customers)
    store_effect = rng.normal(0, 0.35, n_stores)
    product_effect = rng.normal(0, 0.18, n_products)

    monthly_frames = []
    for month in range(n_months):
        n = n_customers
        store = np.where(
            rng.random(n) < 0.88,
            home_store,
            rng.integers(0, n_stores, n),
        )
        product = rng.integers(0, n_products, n)
        channel = rng.choice(["app", "web", "store", "call"], n, p=[0.35, 0.28, 0.30, 0.07])
        season = np.sin(2 * np.pi * month / 12)

        orders = rng.poisson(np.clip(3.2 - 0.55 * latent_risk + 0.25 * season, 0.3, None))
        app_sessions = rng.poisson(np.clip(5.0 - 0.70 * latent_risk + 2.0 * (channel == "app"), 0.3, None))
        days_since = np.clip(
            rng.gamma(2.0, 8.5, n) + 5.5 * latent_risk - 1.7 * orders + 1.2 * month,
            0,
            120,
        )
        support_tickets = rng.poisson(np.clip(0.55 + 0.28 * latent_risk + 0.05 * month, 0.05, None))
        payment_failures = rng.binomial(2, sigmoid_np(-2.7 + 0.55 * latent_risk + 0.04 * month))
        discount_share = np.clip(
            rng.beta(2.2, 5.5, n) + 0.04 * (segment == "value") + 0.015 * month,
            0,
            1,
        )
        satisfaction = np.clip(
            4.1 - 0.35 * support_tickets - 0.25 * payment_failures - 0.25 * latent_risk + rng.normal(0, 0.55, n),
            1,
            5,
        )
        monthly_spend = np.clip(
            base_spend * (1 + 0.12 * season + 0.08 * (segment == "premium") + 0.13 * (segment == "business"))
            + rng.normal(0, 14, n),
            5,
            350,
        )
        tenure = base_tenure + month

        churn_logit = (
            -2.25
            + 0.032 * (days_since - 22)
            - 0.18 * orders
            - 0.055 * app_sessions
            + 0.38 * support_tickets
            + 0.75 * payment_failures
            - 0.32 * (satisfaction - 3.5)
            + 0.55 * latent_risk
            + store_effect[store]
            + product_effect[product]
            + 0.045 * month
            + 0.95 * ((days_since > 42) & (orders <= 1))
            + 0.85 * ((support_tickets >= 2) & (satisfaction < 3.2))
            + 0.65 * ((discount_share > 0.55) & (monthly_spend < 55))
            + 0.55 * ((channel == "call") & (support_tickets >= 1))
            - 0.50 * ((segment == "premium") & (orders >= 4))
        )
        churn_probability = sigmoid_np(churn_logit)
        churn = rng.binomial(1, churn_probability)

        # These variables become available only after the outcome window.
        case_closed_after_outcome = np.where(
            churn == 1,
            rng.binomial(1, 0.94, n),
            rng.binomial(1, 0.04, n),
        )
        future_refund_amount = np.where(
            churn == 1,
            rng.gamma(2.0, 18.0, n),
            rng.gamma(0.6, 2.0, n),
        )

        frame = pd.DataFrame({
            "customer_id": customer_id,
            "month": month,
            "prediction_time": pd.Timestamp("2024-01-01") + pd.offsets.MonthBegin(month),
            "store_id": [f"S{x:03d}" for x in store],
            "product_category": [f"P{x:02d}" for x in product],
            "channel": channel,
            "segment": segment,
            "tenure_months": tenure,
            "monthly_spend": monthly_spend,
            "orders_30d": orders,
            "days_since_last_purchase": days_since,
            "app_sessions_30d": app_sessions,
            "support_tickets_90d": support_tickets,
            "payment_failures_90d": payment_failures,
            "discount_share": discount_share,
            "satisfaction_score": satisfaction,
            "churn_next_30d": churn,
            "case_closed_after_outcome": case_closed_after_outcome,
            "future_refund_amount": future_refund_amount,
        })
        monthly_frames.append(frame)

    data = pd.concat(monthly_frames, ignore_index=True)

    # Inject realistic missingness after the target is generated.
    missing_plan = {
        "monthly_spend": 0.012,
        "discount_share": 0.018,
        "satisfaction_score": 0.022,
        "channel": 0.010,
    }
    for column, rate in missing_plan.items():
        mask = rng.random(len(data)) < rate
        data.loc[mask, column] = np.nan

    return data


customer_df = generate_customer_panel()
print("Shape:", customer_df.shape)
display(customer_df.head())

In [ ]:
# ============================================================
# 2.3 Corpus-style sanity checks for the business table
# ============================================================
missingness_df = (
    customer_df.isna().mean().sort_values(ascending=False).rename("missing_rate").reset_index().rename(columns={"index": "column"})
)
monthly_target_df = (
    customer_df.groupby("month", as_index=False)
    .agg(n_observations=("churn_next_30d", "size"), churn_rate=("churn_next_30d", "mean"))
)

print("Overall churn rate:", round(customer_df["churn_next_30d"].mean(), 4))
display(missingness_df.head(8))
display(monthly_target_df)

plt.figure(figsize=(7, 4))
plt.plot(monthly_target_df["month"], monthly_target_df["churn_rate"], marker="o")
plt.xlabel("Synthetic month")
plt.ylabel("Churn rate")
plt.title("Outcome prevalence changes over time")
plt.show()

## 3. Tensors, parameter shapes, and activation functions

A tensor is a numerical array. In a tabular mini-batch, rows are observations and columns are features, so the input shape is `(B, d)`. A fully connected layer multiplies that tensor by a weight matrix, adds a bias, and returns a new representation.

Hidden layers require nonlinear activations. Without nonlinearity, several stacked linear layers collapse into one linear transformation. ReLU is a common hidden-layer default, while sigmoid and softmax are typically used to interpret binary and multiclass outputs.

In [ ]:
# ============================================================
# 3.1 Tensor shapes and trainable parameters
# ============================================================
reset_seeds()
B, d, h = 4, 6, 3
X_demo = torch.randn(B, d)
layer = nn.Linear(d, h)
Z_demo = layer(X_demo)
H_demo = torch.relu(Z_demo)

shape_df = pd.DataFrame([
    {"object": "mini-batch X", "shape": tuple(X_demo.shape), "meaning": "B observations by d input features"},
    {"object": "weight matrix W", "shape": tuple(layer.weight.shape), "meaning": "h output units by d input features"},
    {"object": "bias vector b", "shape": tuple(layer.bias.shape), "meaning": "one adjustment per output unit"},
    {"object": "pre-activation Z", "shape": tuple(Z_demo.shape), "meaning": "B observations by h hidden units"},
    {"object": "hidden representation H", "shape": tuple(H_demo.shape), "meaning": "ReLU applied elementwise to Z"},
])

display(shape_df)
print("Trainable parameters in the layer:", count_trainable_parameters(layer))
print("Formula check: h*d + h =", h * d + h)

In [ ]:
# ============================================================
# 3.2 Activation functions
# ============================================================
x = np.linspace(-5, 5, 400)
activation_df = pd.DataFrame({
    "x": x,
    "ReLU": np.maximum(0, x),
    "sigmoid": sigmoid_np(x),
    "tanh": np.tanh(x),
})

plt.figure(figsize=(8, 4))
plt.plot(activation_df["x"], activation_df["ReLU"], label="ReLU")
plt.plot(activation_df["x"], activation_df["sigmoid"], label="sigmoid")
plt.plot(activation_df["x"], activation_df["tanh"], label="tanh")
plt.axhline(0, linewidth=0.8)
plt.xlabel("Input value")
plt.ylabel("Activation output")
plt.title("Common activation functions")
plt.legend()
plt.show()

## 4. The forward pass as function composition

The forward pass is a sequence of ordinary numerical operations. The example below computes a two-layer network manually and then confirms that PyTorch returns the same result. Depth adds stages of transformation, which allows later layers to operate on learned intermediate features rather than only on the original columns.

In [ ]:
# ============================================================
# 4.1 Manual forward pass and equivalent PyTorch module
# ============================================================
reset_seeds()
X_small = torch.tensor([[0.5, -1.0], [1.5, 0.2]], dtype=torch.float32)
W1 = torch.tensor([[0.8, -0.4], [0.3, 0.7], [-0.5, 0.6]], dtype=torch.float32)
b1 = torch.tensor([0.1, -0.2, 0.05], dtype=torch.float32)
W2 = torch.tensor([[0.9, -0.7, 0.4]], dtype=torch.float32)
b2 = torch.tensor([-0.1], dtype=torch.float32)

Z1 = X_small @ W1.T + b1
H1 = torch.relu(Z1)
logits_manual = H1 @ W2.T + b2
prob_manual = torch.sigmoid(logits_manual)

network_equivalent = nn.Sequential(
    nn.Linear(2, 3),
    nn.ReLU(),
    nn.Linear(3, 1),
)
with torch.no_grad():
    network_equivalent[0].weight.copy_(W1)
    network_equivalent[0].bias.copy_(b1)
    network_equivalent[2].weight.copy_(W2)
    network_equivalent[2].bias.copy_(b2)

logits_module = network_equivalent(X_small)

forward_df = pd.DataFrame({
    "observation": [0, 1],
    "manual_logit": logits_manual.squeeze(1).numpy(),
    "module_logit": logits_module.detach().squeeze(1).numpy(),
    "predicted_probability": prob_manual.squeeze(1).numpy(),
})
display(forward_df)
print("Maximum absolute difference:", float(torch.max(torch.abs(logits_manual - logits_module))))

## 5. Output layers and loss functions

The hidden representation can support several tasks, but the final output and loss must match the target. A regression network usually returns an unrestricted numeric value. Binary classification uses one logit and binary cross-entropy. Multiclass classification uses one logit per class and categorical cross-entropy. Modern libraries often combine the activation and loss internally for numerical stability.

In [ ]:
# ============================================================
# 5.1 Regression, binary, and multiclass output contracts
# ============================================================
reset_seeds()

# Regression
regression_output = torch.tensor([102.0, 87.5, 119.2])
regression_target = torch.tensor([100.0, 90.0, 120.0])
regression_loss = nn.MSELoss()(regression_output, regression_target)

# Binary classification: BCEWithLogitsLoss applies sigmoid internally.
binary_logits = torch.tensor([-1.2, 0.3, 1.6])
binary_target = torch.tensor([0.0, 1.0, 1.0])
binary_prob = torch.sigmoid(binary_logits)
binary_loss = nn.BCEWithLogitsLoss()(binary_logits, binary_target)

# Multiclass classification: CrossEntropyLoss applies softmax internally.
multiclass_logits = torch.tensor([[2.1, 0.4, -0.3], [0.2, 1.4, 0.7]])
multiclass_target = torch.tensor([0, 2])
multiclass_prob = torch.softmax(multiclass_logits, dim=1)
multiclass_loss = nn.CrossEntropyLoss()(multiclass_logits, multiclass_target)

output_contract_df = pd.DataFrame([
    {"task": "regression", "final representation": "one unrestricted value", "loss": "mean squared error", "example loss": float(regression_loss)},
    {"task": "binary classification", "final representation": "one logit, sigmoid for probability", "loss": "binary cross-entropy", "example loss": float(binary_loss)},
    {"task": "multiclass classification", "final representation": "K logits, softmax for probabilities", "loss": "categorical cross-entropy", "example loss": float(multiclass_loss)},
])

display(output_contract_df)
print("Binary probabilities:", binary_prob.numpy().round(3))
print("Multiclass probabilities:\n", multiclass_prob.numpy().round(3))
print("Each multiclass row sums to:", multiclass_prob.sum(dim=1).numpy())

## 6. Backpropagation as automatic differentiation

Backpropagation applies the chain rule efficiently through the computational graph. The analyst defines the forward computation and the loss. PyTorch records the operations, computes gradients with `loss.backward()`, and stores each gradient in the corresponding parameter. The optimizer then updates the parameters.

In [ ]:
# ============================================================
# 6.1 Inspect one backward pass and one optimizer update
# ============================================================
reset_seeds()
X_grad = torch.randn(8, 3)
y_grad = torch.tensor([0, 1, 0, 1, 1, 0, 1, 0], dtype=torch.float32)

tiny_net = nn.Sequential(
    nn.Linear(3, 5),
    nn.ReLU(),
    nn.Linear(5, 1),
)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(tiny_net.parameters(), lr=0.10)

optimizer.zero_grad()
logits_before = tiny_net(X_grad).squeeze(1)
loss_before = loss_fn(logits_before, y_grad)
loss_before.backward()

gradient_rows = []
for name, parameter in tiny_net.named_parameters():
    gradient_rows.append({
        "parameter": name,
        "shape": tuple(parameter.shape),
        "gradient_norm": float(parameter.grad.norm()),
    })

display(pd.DataFrame(gradient_rows))
optimizer.step()

with torch.no_grad():
    loss_after = loss_fn(tiny_net(X_grad).squeeze(1), y_grad)

print("Loss before update:", round(float(loss_before), 6))
print("Loss after one SGD update:", round(float(loss_after), 6))

## 7. Leakage-safe preprocessing and classical baselines

The deployment scenario is prediction in future months for an existing customer base. Months 0 through 7 form the training set, months 8 and 9 form the validation set, and months 10 and 11 form the final test set. All imputers, scalers, and encoders are fit on the training period only.

A logistic regression and a shallow decision tree provide useful reference points. The tree here is intentionally compact and should not be interpreted as a substitute for a fully tuned boosted-tree benchmark.

In [ ]:
# ============================================================
# 7.1 Define predictors and the time-respecting split
# ============================================================
TARGET = "churn_next_30d"
NUMERIC_FEATURES = [
    "tenure_months",
    "monthly_spend",
    "orders_30d",
    "days_since_last_purchase",
    "app_sessions_30d",
    "support_tickets_90d",
    "payment_failures_90d",
    "discount_share",
    "satisfaction_score",
    "month",
]
CATEGORICAL_FEATURES = ["store_id", "product_category", "channel", "segment"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
LEAKAGE_COLUMNS = ["case_closed_after_outcome", "future_refund_amount"]

train_mask = customer_df["month"] <= 7
val_mask = customer_df["month"].isin([8, 9])
test_mask = customer_df["month"] >= 10

split_df = pd.DataFrame([
    {"split": "train", "months": "0-7", "n": int(train_mask.sum()), "churn_rate": customer_df.loc[train_mask, TARGET].mean()},
    {"split": "validation", "months": "8-9", "n": int(val_mask.sum()), "churn_rate": customer_df.loc[val_mask, TARGET].mean()},
    {"split": "test", "months": "10-11", "n": int(test_mask.sum()), "churn_rate": customer_df.loc[test_mask, TARGET].mean()},
])

display(split_df)
print("Legitimate feature count before encoding:", len(FEATURES))
print("Excluded post-outcome columns:", LEAKAGE_COLUMNS)

In [ ]:
# ============================================================
# 7.2 Fit preprocessing on training data only
# ============================================================
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", make_dense_ohe()),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
], verbose_feature_names_out=False)

X_train = preprocessor.fit_transform(customer_df.loc[train_mask, FEATURES]).astype(np.float32)
X_val = preprocessor.transform(customer_df.loc[val_mask, FEATURES]).astype(np.float32)
X_test = preprocessor.transform(customer_df.loc[test_mask, FEATURES]).astype(np.float32)

y_train = customer_df.loc[train_mask, TARGET].to_numpy(dtype=np.float32)
y_val = customer_df.loc[val_mask, TARGET].to_numpy(dtype=np.float32)
y_test = customer_df.loc[test_mask, TARGET].to_numpy(dtype=np.float32)

encoded_feature_names = preprocessor.get_feature_names_out()
print("Encoded shapes:", X_train.shape, X_val.shape, X_test.shape)
print("First 20 encoded feature names:")
print(encoded_feature_names[:20])

In [ ]:
# ============================================================
# 7.3 Classical baselines
# ============================================================
baseline_models = {
    "logistic_regression": LogisticRegression(max_iter=1200, random_state=SEED),
    "shallow_decision_tree": DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=40,
        random_state=SEED,
    ),
}

val_prob_by_model = {}
test_prob_by_model = {}
fit_seconds_by_model = {}
result_rows = []

for model_name, model in baseline_models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train.astype(int))
    fit_seconds = time.perf_counter() - start
    val_prob = model.predict_proba(X_val)[:, 1]
    test_prob = model.predict_proba(X_test)[:, 1]

    val_prob_by_model[model_name] = val_prob
    test_prob_by_model[model_name] = test_prob
    fit_seconds_by_model[model_name] = fit_seconds
    result_rows.append(to_result_row(model_name, "validation", y_val, val_prob, fit_seconds=fit_seconds))
    result_rows.append(to_result_row(model_name, "test", y_test, test_prob, fit_seconds=fit_seconds))

baseline_results_df = pd.DataFrame(result_rows)
display(baseline_results_df[[
    "model", "split", "roc_auc", "average_precision", "log_loss", "brier_score", "f1", "fit_seconds"
]])

## 8. Multilayer perceptron training with mini-batches and early stopping

The MLP uses two hidden layers with ReLU activations and dropout. `BCEWithLogitsLoss` receives raw logits, and Adam updates the parameters after each mini-batch. Validation loss determines the best checkpoint. Training stops after several epochs without improvement, which limits wasted computation and reduces late-stage overfitting.

In [ ]:
# ============================================================
# 8.1 MLP class and reusable training loop
# ============================================================
class TabularMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=(64, 32), dropout=0.15):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(previous_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)


def fit_tabular_mlp(
    X_train_array,
    y_train_array,
    X_val_array,
    y_val_array,
    hidden_dims=(64, 32),
    dropout=0.15,
    learning_rate=1e-3,
    weight_decay=1e-4,
    max_epochs=TRAIN_EPOCHS,
    patience=5,
    batch_size=BATCH_SIZE,
    restore_best=True,
    verbose=False,
    seed=SEED,
):
    reset_seeds(seed)
    model = TabularMLP(X_train_array.shape[1], hidden_dims, dropout).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )
    loss_function = nn.BCEWithLogitsLoss()

    train_dataset = TensorDataset(
        torch.as_tensor(X_train_array, dtype=torch.float32),
        torch.as_tensor(y_train_array, dtype=torch.float32),
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    X_val_tensor = torch.as_tensor(X_val_array, dtype=torch.float32, device=DEVICE)
    y_val_tensor = torch.as_tensor(y_val_array, dtype=torch.float32, device=DEVICE)

    history = []
    best_state = None
    best_val_loss = np.inf
    best_epoch = 0
    wait = 0
    start = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = loss_function(logits, y_batch)
            loss.backward()
            optimizer.step()
            running_loss += float(loss) * len(X_batch)

        train_loss = running_loss / len(train_dataset)
        model.eval()
        with torch.no_grad():
            val_loss = float(loss_function(model(X_val_tensor), y_val_tensor))

        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if verbose and (epoch == 1 or epoch % 5 == 0):
            print(f"epoch={epoch:02d} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

        if patience is not None and wait >= patience:
            break

    fit_seconds = time.perf_counter() - start
    if restore_best and best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history), {
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_val_loss),
        "epochs_ran": int(len(history)),
        "fit_seconds": float(fit_seconds),
        "parameters": count_trainable_parameters(model),
        "hidden_dims": list(hidden_dims),
        "dropout": float(dropout),
        "learning_rate": float(learning_rate),
        "weight_decay": float(weight_decay),
    }

In [ ]:
# ============================================================
# 8.2 Train the one-hot MLP
# ============================================================
one_hot_mlp, one_hot_history_df, one_hot_run = fit_tabular_mlp(
    X_train,
    y_train,
    X_val,
    y_val,
    hidden_dims=(64, 32),
    dropout=0.15,
    learning_rate=1e-3,
    weight_decay=1e-4,
    max_epochs=TRAIN_EPOCHS,
    patience=5,
    verbose=True,
)

one_hot_val_logits = predict_torch_logits(one_hot_mlp, X_val)
one_hot_test_logits = predict_torch_logits(one_hot_mlp, X_test)
one_hot_val_prob = sigmoid_np(one_hot_val_logits)
one_hot_test_prob = sigmoid_np(one_hot_test_logits)

val_prob_by_model["one_hot_mlp"] = one_hot_val_prob
test_prob_by_model["one_hot_mlp"] = one_hot_test_prob
fit_seconds_by_model["one_hot_mlp"] = one_hot_run["fit_seconds"]

result_rows.append(to_result_row("one_hot_mlp", "validation", y_val, one_hot_val_prob, fit_seconds=one_hot_run["fit_seconds"]))
result_rows.append(to_result_row("one_hot_mlp", "test", y_test, one_hot_test_prob, fit_seconds=one_hot_run["fit_seconds"]))

print(json.dumps(one_hot_run, indent=2))

In [ ]:
# ============================================================
# 8.3 Training and validation loss
# ============================================================
plt.figure(figsize=(7, 4))
plt.plot(one_hot_history_df["epoch"], one_hot_history_df["train_loss"], label="training loss")
plt.plot(one_hot_history_df["epoch"], one_hot_history_df["val_loss"], label="validation loss")
plt.axvline(one_hot_run["best_epoch"], linestyle="--", label="best checkpoint")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.title("Early stopping selects the lowest validation loss")
plt.legend()
plt.show()

current_results_df = pd.DataFrame(result_rows)
display(current_results_df[[
    "model", "split", "roc_auc", "average_precision", "log_loss", "brier_score", "f1", "fit_seconds"
]])

## 9. Categorical embeddings for high-cardinality identifiers

One-hot encoding creates one column per category. An embedding layer instead maps each category ID to a short dense vector that is learned jointly with the prediction task. Embeddings can reduce input width and can capture task-specific similarity among stores, products, customers, or campaigns.

Unknown categories receive index 0. Numeric imputation and scaling are still fit on the training set only.

In [ ]:
# ============================================================
# 9.1 Prepare numeric tensors and categorical index maps
# ============================================================
embedding_numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

Xn_train = embedding_numeric_pipeline.fit_transform(customer_df.loc[train_mask, NUMERIC_FEATURES]).astype(np.float32)
Xn_val = embedding_numeric_pipeline.transform(customer_df.loc[val_mask, NUMERIC_FEATURES]).astype(np.float32)
Xn_test = embedding_numeric_pipeline.transform(customer_df.loc[test_mask, NUMERIC_FEATURES]).astype(np.float32)

category_maps = {}
for column in CATEGORICAL_FEATURES:
    train_values = customer_df.loc[train_mask, column].fillna("<MISSING>").astype(str)
    category_maps[column] = {value: idx + 1 for idx, value in enumerate(sorted(train_values.unique()))}


def encode_category_frame(frame):
    encoded_columns = []
    for column in CATEGORICAL_FEATURES:
        values = frame[column].fillna("<MISSING>").astype(str)
        encoded_columns.append(values.map(category_maps[column]).fillna(0).astype(np.int64).to_numpy())
    return np.stack(encoded_columns, axis=1)


Xc_train = encode_category_frame(customer_df.loc[train_mask])
Xc_val = encode_category_frame(customer_df.loc[val_mask])
Xc_test = encode_category_frame(customer_df.loc[test_mask])
category_cardinalities = [len(category_maps[column]) + 1 for column in CATEGORICAL_FEATURES]

embedding_input_df = pd.DataFrame({
    "feature": CATEGORICAL_FEATURES,
    "cardinality_including_unknown": category_cardinalities,
    "suggested_embedding_dim": [min(16, max(2, int(np.ceil(np.sqrt(c))))) for c in category_cardinalities],
})

display(embedding_input_df)
print("One-hot MLP input width:", X_train.shape[1])
print("Numeric width before embeddings:", Xn_train.shape[1])

In [ ]:
# ============================================================
# 9.2 Embedding MLP and training loop
# ============================================================
class EmbeddingMLP(nn.Module):
    def __init__(self, n_numeric, cardinalities, hidden_dims=(64, 32), dropout=0.15):
        super().__init__()
        self.embedding_dims = [min(16, max(2, int(np.ceil(np.sqrt(c))))) for c in cardinalities]
        self.embeddings = nn.ModuleList([
            nn.Embedding(cardinality, embedding_dim)
            for cardinality, embedding_dim in zip(cardinalities, self.embedding_dims)
        ])
        combined_dim = n_numeric + sum(self.embedding_dims)
        self.network = nn.Sequential(
            nn.Linear(combined_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dims[1], 1),
        )

    def forward(self, numeric_x, categorical_x):
        embedded = [embedding(categorical_x[:, i]) for i, embedding in enumerate(self.embeddings)]
        combined = torch.cat([numeric_x] + embedded, dim=1)
        return self.network(combined).squeeze(1)


def fit_embedding_mlp(max_epochs=TRAIN_EPOCHS, patience=5):
    reset_seeds()
    model = EmbeddingMLP(
        n_numeric=Xn_train.shape[1],
        cardinalities=category_cardinalities,
        hidden_dims=(64, 32),
        dropout=0.15,
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_function = nn.BCEWithLogitsLoss()

    train_dataset = TensorDataset(
        torch.as_tensor(Xn_train, dtype=torch.float32),
        torch.as_tensor(Xc_train, dtype=torch.long),
        torch.as_tensor(y_train, dtype=torch.float32),
    )
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_numeric = torch.as_tensor(Xn_val, dtype=torch.float32, device=DEVICE)
    val_categorical = torch.as_tensor(Xc_val, dtype=torch.long, device=DEVICE)
    val_target = torch.as_tensor(y_val, dtype=torch.float32, device=DEVICE)

    history = []
    best_loss = np.inf
    best_state = None
    best_epoch = 0
    wait = 0
    start = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        running_loss = 0.0
        for numeric_batch, categorical_batch, target_batch in train_loader:
            numeric_batch = numeric_batch.to(DEVICE)
            categorical_batch = categorical_batch.to(DEVICE)
            target_batch = target_batch.to(DEVICE)
            optimizer.zero_grad()
            loss = loss_function(model(numeric_batch, categorical_batch), target_batch)
            loss.backward()
            optimizer.step()
            running_loss += float(loss) * len(target_batch)

        train_loss = running_loss / len(train_dataset)
        model.eval()
        with torch.no_grad():
            val_loss = float(loss_function(model(val_numeric, val_categorical), val_target))
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})

        if val_loss < best_loss - 1e-4:
            best_loss = val_loss
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break

    fit_seconds = time.perf_counter() - start
    model.load_state_dict(best_state)
    run = {
        "best_epoch": int(best_epoch),
        "best_val_loss": float(best_loss),
        "epochs_ran": int(len(history)),
        "fit_seconds": float(fit_seconds),
        "parameters": count_trainable_parameters(model),
        "embedding_dims": model.embedding_dims,
        "combined_input_width": int(Xn_train.shape[1] + sum(model.embedding_dims)),
    }
    return model, pd.DataFrame(history), run


def predict_embedding_model(model, numeric_array, categorical_array, batch_size=1024):
    model.eval()
    dataset = TensorDataset(
        torch.as_tensor(numeric_array, dtype=torch.float32),
        torch.as_tensor(categorical_array, dtype=torch.long),
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    outputs = []
    with torch.no_grad():
        for numeric_batch, categorical_batch in loader:
            logits = model(numeric_batch.to(DEVICE), categorical_batch.to(DEVICE))
            outputs.append(logits.detach().cpu().numpy())
    return np.concatenate(outputs)

In [ ]:
# ============================================================
# 9.3 Train and evaluate the embedding model
# ============================================================
embedding_mlp, embedding_history_df, embedding_run = fit_embedding_mlp()
embedding_val_prob = sigmoid_np(predict_embedding_model(embedding_mlp, Xn_val, Xc_val))
embedding_test_prob = sigmoid_np(predict_embedding_model(embedding_mlp, Xn_test, Xc_test))

val_prob_by_model["embedding_mlp"] = embedding_val_prob
test_prob_by_model["embedding_mlp"] = embedding_test_prob
fit_seconds_by_model["embedding_mlp"] = embedding_run["fit_seconds"]
result_rows.append(to_result_row("embedding_mlp", "validation", y_val, embedding_val_prob, fit_seconds=embedding_run["fit_seconds"]))
result_rows.append(to_result_row("embedding_mlp", "test", y_test, embedding_test_prob, fit_seconds=embedding_run["fit_seconds"]))

representation_comparison_df = pd.DataFrame([
    {
        "representation": "one-hot encoded categories",
        "network_input_width": X_train.shape[1],
        "trainable_parameters": one_hot_run["parameters"],
        "validation_average_precision": safe_ap(y_val, one_hot_val_prob),
        "fit_seconds": one_hot_run["fit_seconds"],
    },
    {
        "representation": "learned categorical embeddings",
        "network_input_width": embedding_run["combined_input_width"],
        "trainable_parameters": embedding_run["parameters"],
        "validation_average_precision": safe_ap(y_val, embedding_val_prob),
        "fit_seconds": embedding_run["fit_seconds"],
    },
])

display(representation_comparison_df)
print(json.dumps(embedding_run, indent=2))

In [ ]:
# ============================================================
# 9.4 Inspect task-specific store proximity in embedding space
# ============================================================
store_embedding = embedding_mlp.embeddings[0].weight.detach().cpu().numpy()
store_map = category_maps["store_id"]
reverse_store_map = {index: value for value, index in store_map.items()}
query_store = "S010" if "S010" in store_map else next(iter(store_map))
query_index = store_map[query_store]

normalized = store_embedding / np.maximum(np.linalg.norm(store_embedding, axis=1, keepdims=True), 1e-9)
similarity = normalized @ normalized[query_index]
ranked_indices = np.argsort(-similarity)

neighbor_rows = []
for index in ranked_indices:
    if index in (0, query_index) or index not in reverse_store_map:
        continue
    neighbor_rows.append({
        "query_store": query_store,
        "neighbor_store": reverse_store_map[index],
        "cosine_similarity": float(similarity[index]),
    })
    if len(neighbor_rows) == 5:
        break

display(pd.DataFrame(neighbor_rows))
print("Interpretation: proximity is learned for this churn task. It is not a universal measure of store similarity.")

## 10. Capacity, regularization, learning rate, and overfitting

Capacity grows with network width, depth, and parameter count. High capacity can fit genuine structure, but it can also memorize a small or noisy training sample. Dropout, L2-style weight decay, smaller architectures, and early stopping are common controls.

The first experiment deliberately trains on only 600 observations and does not restore the best checkpoint. The second experiment changes the learning rate while holding the architecture and data constant.

In [ ]:
# ============================================================
# 10.1 Wide unregularized network versus compact regularized network
# ============================================================
rng = np.random.default_rng(SEED)
small_indices = rng.choice(len(X_train), size=600, replace=False)

wide_model, wide_history_df, wide_run = fit_tabular_mlp(
    X_train[small_indices],
    y_train[small_indices],
    X_val,
    y_val,
    hidden_dims=(256, 128, 64),
    dropout=0.0,
    learning_rate=1e-3,
    weight_decay=0.0,
    max_epochs=35,
    patience=None,
    restore_best=False,
    seed=SEED,
)

regularized_model, regularized_history_df, regularized_run = fit_tabular_mlp(
    X_train[small_indices],
    y_train[small_indices],
    X_val,
    y_val,
    hidden_dims=(32, 16),
    dropout=0.25,
    learning_rate=1e-3,
    weight_decay=1e-3,
    max_epochs=35,
    patience=None,
    restore_best=False,
    seed=SEED,
)

plt.figure(figsize=(8, 4))
plt.plot(wide_history_df["epoch"], wide_history_df["train_loss"], label="wide: train")
plt.plot(wide_history_df["epoch"], wide_history_df["val_loss"], label="wide: validation")
plt.plot(regularized_history_df["epoch"], regularized_history_df["train_loss"], label="regularized: train")
plt.plot(regularized_history_df["epoch"], regularized_history_df["val_loss"], label="regularized: validation")
plt.xlabel("Epoch")
plt.ylabel("Binary cross-entropy")
plt.title("Capacity and regularization on a small training sample")
plt.legend()
plt.show()

capacity_summary_df = pd.DataFrame([
    {
        "model": "wide unregularized",
        "parameters": wide_run["parameters"],
        "minimum_val_loss": wide_history_df["val_loss"].min(),
        "epoch_of_minimum": int(wide_history_df.loc[wide_history_df["val_loss"].idxmin(), "epoch"]),
        "final_train_loss": wide_history_df["train_loss"].iloc[-1],
        "final_val_loss": wide_history_df["val_loss"].iloc[-1],
    },
    {
        "model": "compact regularized",
        "parameters": regularized_run["parameters"],
        "minimum_val_loss": regularized_history_df["val_loss"].min(),
        "epoch_of_minimum": int(regularized_history_df.loc[regularized_history_df["val_loss"].idxmin(), "epoch"]),
        "final_train_loss": regularized_history_df["train_loss"].iloc[-1],
        "final_val_loss": regularized_history_df["val_loss"].iloc[-1],
    },
])
display(capacity_summary_df)

In [ ]:
# ============================================================
# 10.2 Short learning-rate sweep
# ============================================================
learning_rate_histories = {}
learning_rate_rows = []
for learning_rate in [1e-4, 1e-3, 3e-2]:
    _, history_df, run = fit_tabular_mlp(
        X_train[:1200],
        y_train[:1200],
        X_val,
        y_val,
        hidden_dims=(32, 16),
        dropout=0.10,
        learning_rate=learning_rate,
        weight_decay=1e-4,
        max_epochs=8,
        patience=None,
        restore_best=False,
        seed=SEED,
    )
    learning_rate_histories[learning_rate] = history_df
    learning_rate_rows.append({
        "learning_rate": learning_rate,
        "first_val_loss": history_df["val_loss"].iloc[0],
        "best_val_loss": history_df["val_loss"].min(),
        "final_val_loss": history_df["val_loss"].iloc[-1],
    })

plt.figure(figsize=(7, 4))
for learning_rate, history_df in learning_rate_histories.items():
    plt.plot(history_df["epoch"], history_df["val_loss"], marker="o", label=f"lr={learning_rate:g}")
plt.xlabel("Epoch")
plt.ylabel("Validation loss")
plt.title("Learning rate controls the size and stability of updates")
plt.legend()
plt.show()

display(pd.DataFrame(learning_rate_rows))

## 11. Data leakage laboratory

Deep models can exploit weak leakage signals, but leakage is not unique to deep learning. It is a workflow failure. The examples below demonstrate target leakage, group leakage, and preprocessing leakage.

The correct question is always: would this information and this fitted transformation exist at the moment the production prediction is made?

In [ ]:
# ============================================================
# 11.1 Target leakage: post-outcome variables create implausible performance
# ============================================================
leaky_numeric_features = NUMERIC_FEATURES + LEAKAGE_COLUMNS
leaky_features = leaky_numeric_features + CATEGORICAL_FEATURES

leaky_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), leaky_numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", make_dense_ohe()),
    ]), CATEGORICAL_FEATURES),
], verbose_feature_names_out=False)

X_train_leaky = leaky_preprocessor.fit_transform(customer_df.loc[train_mask, leaky_features])
X_test_leaky = leaky_preprocessor.transform(customer_df.loc[test_mask, leaky_features])
leaky_model = LogisticRegression(max_iter=1200, random_state=SEED)
leaky_model.fit(X_train_leaky, y_train.astype(int))
leaky_test_prob = leaky_model.predict_proba(X_test_leaky)[:, 1]

leakage_comparison_df = pd.DataFrame([
    {
        "feature policy": "legitimate predictors only",
        **probability_metrics(y_test, test_prob_by_model["logistic_regression"]),
    },
    {
        "feature policy": "includes post-outcome variables",
        **probability_metrics(y_test, leaky_test_prob),
    },
])
display(leakage_comparison_df[["feature policy", "roc_auc", "average_precision", "log_loss", "brier_score"]])

In [ ]:
# ============================================================
# 11.2 Group leakage: repeated customers and the wrong split boundary
# ============================================================
all_indices = np.arange(len(customer_df))
random_train_idx, random_test_idx = train_test_split(
    all_indices,
    test_size=0.25,
    random_state=SEED,
    stratify=customer_df[TARGET],
)

group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
group_train_idx, group_test_idx = next(group_splitter.split(
    customer_df,
    customer_df[TARGET],
    groups=customer_df["customer_id"],
))


def evaluate_id_memorization(train_idx, test_idx):
    pipeline = Pipeline([
        ("id_encoder", OneHotEncoder(handle_unknown="ignore")),
        ("model", LogisticRegression(max_iter=1200, C=0.7, random_state=SEED)),
    ])
    pipeline.fit(customer_df.iloc[train_idx][["customer_id"]], customer_df.iloc[train_idx][TARGET])
    prob = pipeline.predict_proba(customer_df.iloc[test_idx][["customer_id"]])[:, 1]
    return probability_metrics(customer_df.iloc[test_idx][TARGET], prob)


random_id_metrics = evaluate_id_memorization(random_train_idx, random_test_idx)
group_id_metrics = evaluate_id_memorization(group_train_idx, group_test_idx)

group_leakage_df = pd.DataFrame([
    {"evaluation boundary": "random rows, same customers can repeat", **random_id_metrics},
    {"evaluation boundary": "customer-group split, all test customers are new", **group_id_metrics},
])
display(group_leakage_df[["evaluation boundary", "roc_auc", "average_precision", "log_loss"]])
print("A customer-level split is required when deployment concerns previously unseen customers.")

In [ ]:
# ============================================================
# 11.3 Preprocessing leakage and a documented audit
# ============================================================
train_only_median = customer_df.loc[train_mask, "monthly_spend"].median()
full_data_median = customer_df["monthly_spend"].median()
train_only_mean = customer_df.loc[train_mask, "monthly_spend"].mean()
full_data_mean = customer_df["monthly_spend"].mean()

preprocessing_boundary_df = pd.DataFrame([
    {"statistic": "monthly_spend median", "training data only": train_only_median, "full dataset": full_data_median},
    {"statistic": "monthly_spend mean", "training data only": train_only_mean, "full dataset": full_data_mean},
])

audit_df = pd.DataFrame([
    {"leakage pathway": "temporal leakage", "audit question": "Are all predictors available at prediction time?", "current workflow": "pass: post-outcome fields excluded"},
    {"leakage pathway": "target leakage", "audit question": "Does any feature directly encode churn?", "current workflow": "pass: leakage columns reserved for demonstration"},
    {"leakage pathway": "split mismatch", "audit question": "Does the split match future-month deployment?", "current workflow": "pass: train, validation, and test are ordered by month"},
    {"leakage pathway": "group leakage", "audit question": "Would the system face new customers or known customers?", "current workflow": "documented: main scenario predicts future months for known customers"},
    {"leakage pathway": "preprocessing leakage", "audit question": "Were transformations fit on training data only?", "current workflow": "pass: imputer, scaler, and encoder fit on training months"},
    {"leakage pathway": "test contamination", "audit question": "Was the test set used for tuning?", "current workflow": "pass: validation controls architecture and stopping"},
])

display(preprocessing_boundary_df)
display(audit_df)

## 12. Calibration, business thresholds, and slice evaluation

Model selection should begin with validation performance and operational cost. The code first compares all models on the validation period. It then examines reliability and chooses a threshold that maximizes a simple retention-value function. The selected threshold is locked before final test evaluation.

The numbers in the value function are illustrative. A real project should replace them with estimated treatment effects, contribution margin, intervention cost, and the cost of missed churn.

In [ ]:
# ============================================================
# 12.1 Consolidate results and select by validation average precision
# ============================================================
model_results_df = pd.DataFrame(result_rows)
validation_ranking_df = (
    model_results_df[model_results_df["split"] == "validation"]
    .sort_values(["average_precision", "log_loss"], ascending=[False, True])
    .reset_index(drop=True)
)
selected_model_name = validation_ranking_df.loc[0, "model"]
selected_val_prob = val_prob_by_model[selected_model_name]
selected_test_prob = test_prob_by_model[selected_model_name]

print("Selected model based on validation average precision:", selected_model_name)
display(validation_ranking_df[[
    "model", "roc_auc", "average_precision", "log_loss", "brier_score", "fit_seconds"
]])

comparison_message = (
    "A neural network is justified only when its incremental value exceeds its added cost. "
    "On a structured tabular problem, a simpler baseline may remain the preferred deployment choice."
)
print(comparison_message)

In [ ]:
# ============================================================
# 12.2 Reliability diagram and calibration summary
# ============================================================
fraction_positive, mean_predicted = calibration_curve(y_val, selected_val_prob, n_bins=10, strategy="quantile")

plt.figure(figsize=(5, 5))
plt.plot([0, 1], [0, 1], linestyle="--", label="perfect calibration")
plt.plot(mean_predicted, fraction_positive, marker="o", label=selected_model_name)
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed churn rate")
plt.title("Validation reliability diagram")
plt.legend()
plt.show()

calibration_summary_df = pd.DataFrame([
    {
        "model": selected_model_name,
        "split": "validation",
        "brier_score": brier_score_loss(y_val, selected_val_prob),
        "expected_calibration_error": expected_calibration_error(y_val, selected_val_prob),
        "mean_predicted_probability": selected_val_prob.mean(),
        "observed_rate": y_val.mean(),
    },
    {
        "model": selected_model_name,
        "split": "test",
        "brier_score": brier_score_loss(y_test, selected_test_prob),
        "expected_calibration_error": expected_calibration_error(y_test, selected_test_prob),
        "mean_predicted_probability": selected_test_prob.mean(),
        "observed_rate": y_test.mean(),
    },
])
display(calibration_summary_df)

In [ ]:
# ============================================================
# 12.3 Select a retention threshold on validation value
# ============================================================
TP_VALUE = 45.0
FP_COST = 8.0
FN_COST = 55.0

validation_utility_df = utility_curve(
    y_val,
    selected_val_prob,
    tp_value=TP_VALUE,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
)
best_utility_row = validation_utility_df.loc[validation_utility_df["net_value"].idxmax()]
BUSINESS_THRESHOLD = float(best_utility_row["threshold"])

plt.figure(figsize=(7, 4))
plt.plot(validation_utility_df["threshold"], validation_utility_df["net_value_per_100"])
plt.axvline(BUSINESS_THRESHOLD, linestyle="--", label=f"selected={BUSINESS_THRESHOLD:.2f}")
plt.xlabel("Retention threshold")
plt.ylabel("Validation net value per 100 customers")
plt.title("Threshold selected from an explicit value function")
plt.legend()
plt.show()

threshold_comparison_df = pd.DataFrame([
    {"policy": "default threshold", **probability_metrics(y_test, selected_test_prob, threshold=0.50)},
    {"policy": "validation value threshold", **probability_metrics(y_test, selected_test_prob, threshold=BUSINESS_THRESHOLD)},
])

test_utility_df = utility_curve(
    y_test,
    selected_test_prob,
    tp_value=TP_VALUE,
    fp_cost=FP_COST,
    fn_cost=FN_COST,
)
nearest_test_utility = test_utility_df.iloc[(test_utility_df["threshold"] - BUSINESS_THRESHOLD).abs().argsort()[:1]]

display(best_utility_row.to_frame("validation selection"))
display(threshold_comparison_df[["policy", "threshold", "precision", "recall", "f1", "predicted_positive_rate"]])
display(nearest_test_utility[["threshold", "net_value", "net_value_per_100", "action_rate"]])

In [ ]:
# ============================================================
# 12.4 Slice evaluation at the locked business threshold
# ============================================================
test_rows = customer_df.loc[test_mask].copy()
test_rows["predicted_probability"] = selected_test_prob
test_rows["predicted_action"] = (selected_test_prob >= BUSINESS_THRESHOLD).astype(int)

slice_rows = []
for segment_name, group in test_rows.groupby("segment", dropna=False):
    y_group = group[TARGET].to_numpy()
    p_group = group["predicted_probability"].to_numpy()
    metrics = probability_metrics(y_group, p_group, BUSINESS_THRESHOLD)
    slice_rows.append({
        "segment": str(segment_name),
        "n": len(group),
        "observed_churn_rate": y_group.mean(),
        **metrics,
    })

slice_metrics_df = pd.DataFrame(slice_rows).sort_values("segment")
display(slice_metrics_df[[
    "segment", "n", "observed_churn_rate", "roc_auc", "average_precision", "precision", "recall", "predicted_positive_rate"
]])

## 13. Governance, experiment records, and saved artifacts

A reproducible deep-learning experiment records the data boundary, target, feature policy, architecture, optimizer settings, checkpoint rule, metrics, threshold policy, and known limitations. Model files without this context are difficult to audit and risky to reuse.

In [ ]:
# ============================================================
# 13.1 Experiment log and system card
# ============================================================
experiment_log_df = model_results_df.copy()
experiment_log_df["selected_for_thresholding"] = experiment_log_df["model"].eq(selected_model_name)

system_card = {
    "chapter": 14,
    "system_name": "synthetic_monthly_churn_risk",
    "business_decision": "prioritize customers for a retention intervention",
    "unit_of_analysis": "one customer at one monthly prediction time",
    "target": TARGET,
    "prediction_horizon": "next 30 days",
    "legitimate_features": FEATURES,
    "excluded_post_outcome_features": LEAKAGE_COLUMNS,
    "split_strategy": {
        "train": "months 0-7",
        "validation": "months 8-9",
        "test": "months 10-11",
    },
    "preprocessing": "median imputation and scaling for numeric features; most-frequent imputation and one-hot encoding for categorical features; fit on training data only",
    "models_compared": sorted(val_prob_by_model.keys()),
    "selected_model": selected_model_name,
    "selection_metric": "validation average precision, with log loss as a secondary check",
    "business_threshold": BUSINESS_THRESHOLD,
    "threshold_values": {
        "true_positive_value": TP_VALUE,
        "false_positive_cost": FP_COST,
        "false_negative_cost": FN_COST,
    },
    "test_metrics_at_business_threshold": probability_metrics(y_test, selected_test_prob, BUSINESS_THRESHOLD),
    "leakage_controls": audit_df.to_dict(orient="records"),
    "known_limitations": [
        "the data are synthetic and simpler than a production customer system",
        "the main split assumes future predictions for an existing customer base",
        "retention value is illustrative and does not estimate causal treatment effects",
        "store embeddings reflect this churn objective and should not be reused as universal similarity measures",
    ],
}

display(experiment_log_df[[
    "model", "split", "roc_auc", "average_precision", "log_loss", "brier_score", "fit_seconds", "selected_for_thresholding"
]])
print(json.dumps(system_card, indent=2))

In [ ]:
# ============================================================
# 13.2 Save shareable analysis artifacts
# ============================================================
customer_df.to_csv(OUT_DIR / "ch14_synthetic_customer_panel.csv", index=False)
model_results_df.to_csv(OUT_DIR / "ch14_model_results.csv", index=False)
one_hot_history_df.to_csv(OUT_DIR / "ch14_one_hot_mlp_training_history.csv", index=False)
embedding_history_df.to_csv(OUT_DIR / "ch14_embedding_mlp_training_history.csv", index=False)
capacity_summary_df.to_csv(OUT_DIR / "ch14_capacity_experiment.csv", index=False)
validation_utility_df.to_csv(OUT_DIR / "ch14_validation_utility_curve.csv", index=False)
calibration_summary_df.to_csv(OUT_DIR / "ch14_calibration_summary.csv", index=False)
slice_metrics_df.to_csv(OUT_DIR / "ch14_slice_metrics.csv", index=False)
audit_df.to_csv(OUT_DIR / "ch14_leakage_audit.csv", index=False)

joblib.dump(preprocessor, OUT_DIR / "ch14_preprocessor.joblib")
joblib.dump(baseline_models["logistic_regression"], OUT_DIR / "ch14_logistic_regression.joblib")
joblib.dump(baseline_models["shallow_decision_tree"], OUT_DIR / "ch14_shallow_decision_tree.joblib")
joblib.dump(embedding_numeric_pipeline, OUT_DIR / "ch14_embedding_numeric_pipeline.joblib")
joblib.dump(category_maps, OUT_DIR / "ch14_category_maps.joblib")
torch.save(one_hot_mlp.state_dict(), OUT_DIR / "ch14_one_hot_mlp_state.pt")
torch.save(embedding_mlp.state_dict(), OUT_DIR / "ch14_embedding_mlp_state.pt")

with open(OUT_DIR / "ch14_system_card.json", "w", encoding="utf-8") as file:
    json.dump(system_card, file, indent=2)

print("Saved artifacts:")
for path in sorted(OUT_DIR.iterdir()):
    print(" -", path)

## Decision guide

Begin with the business decision and a classical baseline. Use an MLP for tabular data when nonlinear interactions, large samples, categorical embeddings, or future multimodal expansion create a credible advantage. Keep the simpler model when the neural network does not deliver stable incremental value after accounting for compute, tuning, interpretation, and maintenance.

Match the output layer and loss to the target, fit preprocessing on the training set only, monitor both training and validation loss, save the best checkpoint, and reserve the test set for the final report. Treat probability calibration and threshold choice as part of the decision system. A high test score is not credible when the feature timeline, entity boundary, or preprocessing boundary is unclear.

## Exercises

1. Change the hidden widths from `(64, 32)` to `(16, 8)` and `(128, 64, 32)`. Compare parameter count, fit time, validation loss, and test average precision.

2. Set dropout to `0.0`, `0.20`, and `0.50`. Which value creates the best validation loss? Does the answer change when the training sample is reduced?

3. Replace Adam with SGD in `fit_tabular_mlp`. Test several learning rates and explain why the same learning rate does not work equally well for both optimizers.

4. Remove `store_id` from the one-hot model and the embedding model. Does either representation lose more predictive value?

5. Change the target to a synthetic continuous outcome such as next-month spend. Modify the output layer, loss function, and evaluation metrics for regression.

6. Create a three-class service-priority target. Modify the network to return three logits and train it with `CrossEntropyLoss`.

7. Add `case_closed_after_outcome` to the legitimate feature list, observe the performance jump, and write a one-paragraph explanation of why that result must be rejected.

8. Redefine deployment as prediction for entirely new customers. Replace the time split with a customer-group split and compare the model ranking.

9. Increase the false-negative cost in the value function. Explain how the selected threshold, recall, and action rate change.

10. Add one reproducibility field and one operational monitoring field to the system card.

## Wrap-up

This notebook converted the foundations of deep learning into a complete business analytics workflow. It covered tensors, activations, forward computation, output contracts, automatic differentiation, mini-batch training, early stopping, categorical embeddings, capacity control, learning-rate sensitivity, leakage prevention, calibration checks, value-based thresholds, slice evaluation, and experiment governance. The central lesson is not that deep learning should replace classical methods. It is that deep learning becomes useful when its learned representations and nonlinear capacity produce stable business value under an honest evaluation design.